# Gatefall — generate with a trained LoRA on Google Colab (free GPU)

Runs ComfyUI on Colab's free Tesla T4 with your trained character
LoRA (from `gatefall_lora_training.ipynb`) loaded on top of the base
checkpoint, so pose/crop/expression prompts hold the character's
design far more reliably than a bare fixed-seed prompt does once the
composition tokens change.

**Prerequisite:** you need a trained LoRA already saved to
`gatefall-loras/` in your Google Drive — run
`gatefall_lora_training.ipynb` first if you haven't.

**Before running:** `Runtime` -> `Change runtime type` -> `T4 GPU` ->
`Save`. Run cells in order.

**Just want ComfyUI without a LoRA?** Use `gatefall_comfyui.ipynb`
instead — this notebook is specifically for testing/using a trained
LoRA.

**Colab free-tier limits apply** — sessions disconnect after
inactivity and there's a rolling GPU-time cap. Save any image you
want to keep — closing the tab or losing the session wipes the Colab
disk.

## 1. Confirm the GPU is attached

In [ ]:
!nvidia-smi

## 2. Install ComfyUI

In [ ]:
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install -r requirements.txt -q

## 3. Get the base checkpoint

**This must be the exact same checkpoint you trained the LoRA
against** (same model, same architecture family) — a mismatch either
loads incorrectly or produces character results that don't match what
the LoRA actually learned. Check `MODEL_TYPE` and `CHECKPOINT_PATH`
from your training run if you're not sure which one that was.

Two ways to get the checkpoint into Colab — pick **one** of 3a / 3b
and run only that cell.

### 3a. Download directly into Colab from Civitai

1. Go to civitai.com and open the exact model page/version you
   trained against.
2. Right-click the **Download** button -> copy link address
   (looks like `https://civitai.com/api/download/models/XXXXXX`).
3. Paste it into `CHECKPOINT_URL` below.

Some Civitai models require being logged in to download. If the
download fails with an auth error, get an API key from
civitai.com -> account settings -> API Keys, and paste it into
`CIVITAI_TOKEN`. Leave it as an empty string if not needed.

In [ ]:
CHECKPOINT_URL = "https://civitai.com/api/download/models/REPLACE_ME"  # @param {type:"string"}
CIVITAI_TOKEN = ""  # @param {type:"string"}

import os
url = CHECKPOINT_URL
if CIVITAI_TOKEN:
    sep = "&" if "?" in url else "?"
    url = f"{url}{sep}token={CIVITAI_TOKEN}"

os.makedirs("/content/ComfyUI/models/checkpoints", exist_ok=True)
!wget -nc --content-disposition "$url" -P /content/ComfyUI/models/checkpoints
!ls -lh /content/ComfyUI/models/checkpoints

### 3b. Already have it in Google Drive? Mount and symlink instead

Skips re-downloading a multi-GB file. This also mounts Drive, which
step 4 below needs anyway to fetch your trained LoRA.

In [ ]:
DRIVE_CHECKPOINT_PATH = "/content/drive/MyDrive/gatefall-checkpoints/aniversePonyXL_v60.safetensors"  # @param {type:"string"}

from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs("/content/ComfyUI/models/checkpoints", exist_ok=True)
dest = os.path.join("/content/ComfyUI/models/checkpoints", os.path.basename(DRIVE_CHECKPOINT_PATH))
# Symlink instead of copy: instant, and avoids duplicating a multi-GB file onto the Colab disk.
if not os.path.exists(dest):
    os.symlink(DRIVE_CHECKPOINT_PATH, dest)
!ls -lh /content/ComfyUI/models/checkpoints

Either way, confirm a `.safetensors` file of several GB shows up in the
listing above before continuing.

## 4. Get your trained LoRA from Drive

Pulls the LoRA(s) saved by `gatefall_lora_training.ipynb`'s final
step into ComfyUI's `models/loras` folder. If you used 3b above, Drive
is already mounted and this cell just copies the files; if you used
3a, this also mounts Drive.

By default this copies **every** `.safetensors` file in your LoRA
training output folder (all the intermediate epoch checkpoints plus
the final one), so you can compare them side by side in step 6 —
narrow `LORA_GLOB` if you only want a specific file.

In [ ]:
DRIVE_LORA_DIR = "/content/drive/MyDrive/gatefall-loras"  # @param {type:"string"}
LORA_GLOB = "faelen_lora*.safetensors"  # @param {type:"string"}

from google.colab import drive
drive.mount("/content/drive")

import os, glob, shutil
os.makedirs("/content/ComfyUI/models/loras", exist_ok=True)

matches = glob.glob(os.path.join(DRIVE_LORA_DIR, LORA_GLOB))
if not matches:
    print(f"WARNING: no files matched {LORA_GLOB} in {DRIVE_LORA_DIR} — check the path/pattern.")
for f in matches:
    shutil.copy(f, "/content/ComfyUI/models/loras")

!ls -lh /content/ComfyUI/models/loras

## 5. Launch ComfyUI and open it in your browser

### 5a. Colab's built-in port proxy (recommended, try this first)

Uses Google's own infrastructure instead of a third-party tunnel — no
separate service to fail, and it's authenticated to your Google
account automatically.

In [ ]:
%cd /content/ComfyUI

import subprocess, time

comfy_proc = subprocess.Popen(
    ["python3", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    cwd="/content/ComfyUI",
)
time.sleep(15)  # give ComfyUI time to start before opening the proxy

from google.colab.output import eval_js
proxy_url = eval_js("google.colab.kernel.proxyPort(8188)")
print(f"Open ComfyUI here: {proxy_url}")

**Don't click the printed link directly from this page** — copy it and
paste it into a new browser tab's address bar instead. ComfyUI has a
built-in CSRF check that returns `403 Forbidden` whenever it sees a
cross-site navigation, and clicking a link inside the Colab page
counts as one. Pasting the URL directly into a fresh tab's address bar
avoids that.

**This cell keeps running** (it's what keeps ComfyUI alive) — leave it
running while you work, don't interrupt it until you're done for the
session.

### 5b. Fallback: Cloudflare quick tunnel

Only use this if 5a doesn't work for some reason. Run **either** 5a or
5b, not both (both try to launch ComfyUI on the same port — restart
the runtime first if you already ran 5a).

In [ ]:
%cd /content/ComfyUI
!wget -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

import subprocess, time, re

comfy_proc = subprocess.Popen(
    ["python3", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    cwd="/content/ComfyUI",
)
time.sleep(15)  # give ComfyUI time to start before opening the tunnel

tunnel_proc = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8188"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print("Waiting for tunnel URL...")
for line in tunnel_proc.stdout:
    print(line, end="")
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        print(f"\n\nOpen ComfyUI here: {match.group(0)}\n")
        break

Same as 5a: **don't click this link directly** — copy it and paste it
into a new tab's address bar.

## 6. Generate with your LoRA

The default graph (`Load Checkpoint` -> prompts -> `KSampler` ->
`Save Image`) doesn't apply a LoRA yet — you need to insert a
**`LoraLoader`** node:

1. In `Load Checkpoint`, select the checkpoint you added in step 3.
2. Search-add a **`LoraLoader`** node. Wire `Load Checkpoint`'s
   `MODEL` and `CLIP` outputs into `LoraLoader`'s `model`/`clip`
   inputs.
3. Rewire everything downstream to come from `LoraLoader`'s
   `MODEL`/`CLIP` outputs instead of `Load Checkpoint`'s directly —
   that means `KSampler`'s `model` input and both `CLIP Text Encode`
   nodes' `clip` input.
4. In `LoraLoader`, select the LoRA file you added in step 4 (e.g.
   `faelen_lora.safetensors` for the final epoch, or one of the
   numbered `faelen_lora-0000XX.safetensors` intermediate
   checkpoints). Set `strength_model` and `strength_clip` around
   `0.7-0.9` to start.
5. In the positive `CLIP Text Encode` box, include the **trigger
   word** you trained with (e.g. `flnwarden`) — that's what activates
   the learned character identity. Add the rest of your prompt from
   `docs/art-direction.md` around it. If you're on a Pony-based
   checkpoint, keep the `score_9, score_8_up, score_7_up, ` quality
   tag prefix.
6. Use an SDXL-native resolution (1024x1024, or 832x1216 for a
   portrait) in `Empty Latent Image`.
7. Click **Queue Prompt**.

**Comparing epochs:** if you pulled multiple checkpoints in step 4,
swap the file selected in `LoraLoader` and re-queue with the same
prompt/seed to compare them directly — an earlier epoch (e.g.
`-000006`) sometimes generalizes better on a small dataset, while the
final epoch locks identity harder but can look stiffer. Pick whichever
holds the design best without over-fitting the training images.

Generated images save to `/content/ComfyUI/output/` on the Colab VM —
download anything you want to keep before the session ends.

## 7. (Optional) Download all outputs as a zip

In [ ]:
from google.colab import files
!zip -r /content/gatefall_outputs.zip /content/ComfyUI/output
files.download("/content/gatefall_outputs.zip")